# NB09 · Benchmark multisemilla YOLO11 vs YOLO26 — palma africana

**Proyecto:** PALMA_AFRICANA_YOLO  
**Caso del libro:** clasificación de madurez y condición del fruto  
**Entorno obligatorio:** GPU NVIDIA  
**Recomendado:** GPU L4 de Google Colab  
**Tiempo estimado:** L4, 20–35 min; T4, 35–55 min  
**CPU:** no recomendada para entrenamiento (aprox. 3–6 h)

El notebook amplía el experimento NB04 sin repetirlo:

- reutiliza las nueve corridas YOLO11 (n, s y m; semillas 2026–2028);
- entrena nueve corridas equivalentes con YOLO26;
- mantiene 384 px, 60 épocas máximas y paciencia 15;
- selecciona con F1 macro promedio de validación, sin usar prueba para elegir;
- produce tablas y figuras profesionales a 300 dpi para el capítulo 7.

> El conjunto de prueba se emplea únicamente para estimar la generalización final.


In [ ]:
%pip install -q ultralytics==8.4.172 scikit-learn seaborn


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import gc
import json
import platform
import random
import shutil
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import ultralytics
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
)
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó GPU. En Colab seleccione Entorno de ejecución > '
        'Cambiar tipo de entorno de ejecución > GPU L4 o T4.'
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {GPU_NAME} ({GPU_MEMORY_GB:.1f} GB)')
print('PyTorch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)


## 1. Rutas, modelos y protocolo experimental


In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
DRIVE_DATA = PROJECT_ROOT / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
MODELS_ROOT = PROJECT_ROOT / '05_MODELOS'
RESULTS_ROOT = PROJECT_ROOT / '06_RESULTADOS'
NB04_RESULTS = RESULTS_ROOT / 'NB04_COMPARACION_YOLO11_MULTISEED'
NB09_RESULTS = RESULTS_ROOT / 'NB09_BENCHMARK_YOLO11_YOLO26_MULTISEED'

LOCAL_DATA = Path('/content/PALMA_CLS_NB09')
LOCAL_RUNS = Path('/content/palma_nb09_runs')
CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

EPOCHS = 60
IMGSZ = 384
PATIENCE = 15
SEEDS = [2026, 2027, 2028]

EXPERIMENTS = [
    {'model_key': 'yolo26n', 'weights': 'yolo26n-cls.pt', 'batch': 16, 'seed': seed}
    for seed in SEEDS
] + [
    {'model_key': 'yolo26s', 'weights': 'yolo26s-cls.pt', 'batch': 16, 'seed': seed}
    for seed in SEEDS
] + [
    {'model_key': 'yolo26m', 'weights': 'yolo26m-cls.pt', 'batch': 8, 'seed': seed}
    for seed in SEEDS
]

for path in [NB09_RESULTS, MODELS_ROOT, LOCAL_RUNS]:
    path.mkdir(parents=True, exist_ok=True)

required = [DRIVE_DATA, NB04_RESULTS / 'resultados_por_corrida.csv']
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Faltan resultados de NB04 o el dataset procesado:\n' + '\n'.join(missing))

display(pd.DataFrame(EXPERIMENTS))


## 2. Copia local y control del conjunto de datos


In [ ]:
from google.colab import drive as colab_drive

MAX_COPY_ATTEMPTS = 5


def reconnect_drive():
    print('Reconectando Google Drive...')
    try:
        colab_drive.mount('/content/drive', force_remount=True)
    except Exception as exc:
        print('Primer intento de reconexión:', type(exc).__name__, str(exc)[:160])
        time.sleep(3)
        colab_drive.mount('/content/drive', force_remount=True)


def ensure_source_ready():
    required = [DRIVE_DATA / split / label for split in ['train', 'val', 'test'] for label in CLASSES]
    if all(path.exists() for path in required):
        return
    reconnect_drive()
    if not all(path.exists() for path in required):
        missing = [str(path) for path in required if not path.exists()]
        raise FileNotFoundError('Drive no quedó disponible. Faltan:\n' + '\n'.join(missing))


def list_source_files(source_dir):
    for attempt in range(1, MAX_COPY_ATTEMPTS + 1):
        try:
            return sorted(
                path for path in source_dir.iterdir()
                if path.is_file() and path.suffix.lower() in IMAGE_EXTS
            )
        except OSError as exc:
            print(f'Lectura de Drive falló ({attempt}/{MAX_COPY_ATTEMPTS}): {exc}')
            reconnect_drive()
            time.sleep(2)
    raise RuntimeError(f'No se pudo leer {source_dir} después de varios intentos.')


def copy_with_retry(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, MAX_COPY_ATTEMPTS + 1):
        try:
            if destination.exists() and destination.stat().st_size == source.stat().st_size:
                return False
            temporary = destination.with_suffix(destination.suffix + '.part')
            if temporary.exists():
                temporary.unlink()
            shutil.copy2(source, temporary)
            temporary.replace(destination)
            return True
        except OSError as exc:
            print(f'Copia falló ({attempt}/{MAX_COPY_ATTEMPTS}) · {source.name}: {exc}')
            reconnect_drive()
            time.sleep(2 * attempt)
    raise RuntimeError(f'No se pudo copiar: {source}')


ensure_source_ready()
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

copied = 0
reused = 0
for split in ['train', 'val', 'test']:
    for label in CLASSES:
        source_dir = DRIVE_DATA / split / label
        destination_dir = LOCAL_DATA / split / label
        destination_dir.mkdir(parents=True, exist_ok=True)
        source_files = list_source_files(source_dir)
        for source in source_files:
            destination = destination_dir / source.name
            if copy_with_retry(source, destination):
                copied += 1
            else:
                reused += 1
        print(f'{split}/{label}: {len(source_files)} imágenes listas')

distribution_rows = []
for split in ['train', 'val', 'test']:
    for label in CLASSES:
        folder = LOCAL_DATA / split / label
        n = sum(1 for path in folder.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTS)
        distribution_rows.append({'split': split, 'class': label, 'n': n})

distribution = pd.DataFrame(distribution_rows)
observed = distribution.groupby('split')['n'].sum().to_dict()
expected = {'train': 312, 'val': 70, 'test': 70}
if observed != expected:
    raise RuntimeError(f'Copia local incompleta. Esperado: {expected}; observado: {observed}')

display(distribution.pivot(index='class', columns='split', values='n'))
print(f'Datos locales listos: {sum(observed.values())} imágenes')
print(f'Copiadas ahora: {copied} | Reutilizadas: {reused}')


## 3. Funciones de evaluación reproducible


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def split_paths(split):
    return sorted(
        path for path in (LOCAL_DATA / split).glob('*/*')
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )


def evaluate_model(model, split, imgsz, batch, run_name, output_dir):
    paths = split_paths(split)
    start = time.perf_counter()
    outputs = model.predict(
        source=[str(path) for path in paths],
        imgsz=imgsz,
        batch=batch,
        device=0,
        verbose=False,
    )
    seconds = time.perf_counter() - start

    rows = []
    for path, result in zip(paths, outputs):
        probabilities = result.probs.data.detach().cpu().numpy().astype(float)
        names = {int(key): value for key, value in result.names.items()}
        pred_id = int(np.argmax(probabilities))
        row = {
            'run_name': run_name,
            'split': split,
            'image': path.name,
            'relative_path': str(path.relative_to(LOCAL_DATA)),
            'true_class': path.parent.name,
            'pred_class': names[pred_id],
            'confidence': float(probabilities[pred_id]),
            'correct': bool(path.parent.name == names[pred_id]),
        }
        for class_id, probability in enumerate(probabilities):
            row[f'prob_{names[class_id]}'] = float(probability)
        rows.append(row)

    frame = pd.DataFrame(rows)
    y_true, y_pred = frame['true_class'], frame['pred_class']
    report = classification_report(
        y_true, y_pred, labels=CLASSES, target_names=CLASSES,
        output_dict=True, zero_division=0,
    )
    metrics = {
        f'{split}_n': int(len(frame)),
        f'{split}_accuracy': float(accuracy_score(y_true, y_pred)),
        f'{split}_balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        f'{split}_precision_macro': float(precision_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
        f'{split}_recall_macro': float(recall_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
        f'{split}_f1_macro': float(f1_score(y_true, y_pred, labels=CLASSES, average='macro', zero_division=0)),
        f'{split}_f1_weighted': float(f1_score(y_true, y_pred, labels=CLASSES, average='weighted', zero_division=0)),
        f'{split}_inference_ms_image': float(seconds * 1000 / len(frame)),
    }
    for label in CLASSES:
        metrics[f'{split}_f1_{label}'] = float(report[label]['f1-score'])

    frame.to_csv(output_dir / f'predicciones_{split}.csv', index=False)
    pd.DataFrame(report).T.to_csv(output_dir / f'reporte_{split}.csv')
    pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CLASSES),
        index=CLASSES, columns=CLASSES,
    ).to_csv(output_dir / f'matriz_confusion_{split}.csv')
    return metrics


## 4. Entrenamiento reanudable de las nueve corridas YOLO26


In [ ]:
def run_experiment(exp):
    model_key = exp['model_key']
    seed = exp['seed']
    batch = exp['batch']
    run_name = f'{model_key}_cls_seed{seed}'
    run_results = NB09_RESULTS / run_name
    run_model_dir = MODELS_ROOT / run_name
    summary_path = run_results / 'resumen.json'
    saved_best = run_model_dir / 'best.pt'
    run_results.mkdir(parents=True, exist_ok=True)
    run_model_dir.mkdir(parents=True, exist_ok=True)

    if summary_path.exists() and saved_best.exists():
        print(f'REUTILIZADA: {run_name}')
        return json.loads(summary_path.read_text(encoding='utf-8'))

    set_seed(seed)
    print(f'ENTRENANDO: {run_name}')
    local_run = LOCAL_RUNS / run_name
    start = time.time()
    trainer = YOLO(exp['weights'])
    trainer.train(
        data=str(LOCAL_DATA), epochs=EPOCHS, imgsz=IMGSZ,
        batch=batch, patience=PATIENCE, device=0, workers=2,
        seed=seed, deterministic=True, pretrained=True,
        optimizer='auto', amp=True, cache=True,
        project=str(LOCAL_RUNS), name=run_name, exist_ok=True,
        plots=True, verbose=False,
    )
    training_minutes = (time.time() - start) / 60
    best_path = local_run / 'weights' / 'best.pt'
    if not best_path.exists():
        raise FileNotFoundError(f'No se generó {best_path}')

    shutil.copy2(best_path, saved_best)
    for item in ['results.csv', 'results.png', 'confusion_matrix.png',
                 'confusion_matrix_normalized.png', 'args.yaml']:
        source = local_run / item
        if source.exists():
            shutil.copy2(source, run_model_dir / item)

    model_size_mb = saved_best.stat().st_size / 1024**2
    best_model = YOLO(str(saved_best))
    val_metrics = evaluate_model(best_model, 'val', IMGSZ, batch, run_name, run_results)
    test_metrics = evaluate_model(best_model, 'test', IMGSZ, batch, run_name, run_results)

    summary = {
        'run_name': run_name,
        'family': 'YOLO26',
        'model_key': model_key,
        'size': model_key[-1],
        'weights': exp['weights'],
        'seed': seed,
        'batch': batch,
        'imgsz': IMGSZ,
        'epochs_max': EPOCHS,
        'patience': PATIENCE,
        'gpu': GPU_NAME,
        'training_minutes': float(training_minutes),
        'model_size_mb': float(model_size_mb),
        **val_metrics,
        **test_metrics,
    }
    summary_path.write_text(
        json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(
        f"FINALIZADA: {run_name} | val F1={summary['val_f1_macro']:.4f} "
        f"| test F1={summary['test_f1_macro']:.4f}"
    )

    del best_model, trainer
    torch.cuda.empty_cache()
    gc.collect()
    return summary


In [ ]:
yolo26_summaries = []
for position, exp in enumerate(EXPERIMENTS, start=1):
    print(f'\n=== Corrida YOLO26 {position}/{len(EXPERIMENTS)} ===')
    summary = run_experiment(exp)
    yolo26_summaries.append(summary)
    pd.DataFrame(yolo26_summaries).to_csv(
        NB09_RESULTS / 'resultados_YOLO26_parciales.csv', index=False
    )

yolo26_runs = pd.DataFrame(yolo26_summaries).sort_values(['model_key', 'seed']).reset_index(drop=True)
yolo26_runs.to_csv(NB09_RESULTS / 'resultados_YOLO26_por_corrida.csv', index=False)
display(yolo26_runs[[
    'model_key', 'seed', 'val_accuracy', 'val_f1_macro',
    'test_accuracy', 'test_f1_macro', 'training_minutes',
    'model_size_mb', 'test_inference_ms_image',
]].round(4))


## 5. Integración de YOLO11 y YOLO26


In [ ]:
yolo11_runs = pd.read_csv(NB04_RESULTS / 'resultados_por_corrida.csv').copy()
yolo11_runs['family'] = 'YOLO11'
yolo11_runs['size'] = yolo11_runs['model_key'].str[-1]

if 'family' not in yolo26_runs.columns:
    yolo26_runs['family'] = 'YOLO26'
if 'size' not in yolo26_runs.columns:
    yolo26_runs['size'] = yolo26_runs['model_key'].str[-1]

common_columns = sorted(set(yolo11_runs.columns) & set(yolo26_runs.columns))
all_runs = pd.concat(
    [yolo11_runs[common_columns], yolo26_runs[common_columns]],
    ignore_index=True,
)
all_runs['architecture'] = all_runs['family'] + all_runs['size']
all_runs.to_csv(NB09_RESULTS / 'benchmark_YOLO11_YOLO26_por_corrida.csv', index=False)

aggregate = all_runs.groupby(['family', 'size', 'architecture']).agg(
    val_accuracy_mean=('val_accuracy', 'mean'),
    val_accuracy_std=('val_accuracy', 'std'),
    val_f1_macro_mean=('val_f1_macro', 'mean'),
    val_f1_macro_std=('val_f1_macro', 'std'),
    test_accuracy_mean=('test_accuracy', 'mean'),
    test_accuracy_std=('test_accuracy', 'std'),
    test_f1_macro_mean=('test_f1_macro', 'mean'),
    test_f1_macro_std=('test_f1_macro', 'std'),
    training_minutes_mean=('training_minutes', 'mean'),
    model_size_mb_mean=('model_size_mb', 'mean'),
    inference_ms_image_mean=('test_inference_ms_image', 'mean'),
).reset_index()
aggregate = aggregate.sort_values('val_f1_macro_mean', ascending=False).reset_index(drop=True)
aggregate.insert(0, 'validation_rank', np.arange(1, len(aggregate) + 1))
aggregate.to_csv(NB09_RESULTS / 'benchmark_resumen_por_modelo.csv', index=False)

family_summary = all_runs.groupby('family').agg(
    val_f1_macro_mean=('val_f1_macro', 'mean'),
    val_f1_macro_std=('val_f1_macro', 'std'),
    test_f1_macro_mean=('test_f1_macro', 'mean'),
    test_f1_macro_std=('test_f1_macro', 'std'),
    inference_ms_image_mean=('test_inference_ms_image', 'mean'),
).reset_index()
family_summary.to_csv(NB09_RESULTS / 'benchmark_resumen_por_familia.csv', index=False)

display(aggregate.round(4))
display(family_summary.round(4))


## 6. Figuras profesionales para el capítulo 7


In [ ]:
sns.set_theme(style='whitegrid', context='talk')
palette = {'YOLO11': '#1F5C3D', 'YOLO26': '#D18B2C'}
order = ['YOLO11n', 'YOLO26n', 'YOLO11s', 'YOLO26s', 'YOLO11m', 'YOLO26m']

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), constrained_layout=True)
sns.boxplot(
    data=all_runs, x='architecture', y='val_f1_macro', order=order,
    hue='family', palette=palette, dodge=False, ax=axes[0], width=0.65,
)
sns.stripplot(
    data=all_runs, x='architecture', y='val_f1_macro', order=order,
    color='#222222', size=5, jitter=0.08, ax=axes[0],
)
axes[0].set_title('Validación · selección del modelo')
axes[0].set_xlabel('Arquitectura')
axes[0].set_ylabel('F1 macro')
axes[0].set_ylim(0, 1.03)
if axes[0].legend_:
    axes[0].legend_.remove()

sns.boxplot(
    data=all_runs, x='architecture', y='test_f1_macro', order=order,
    hue='family', palette=palette, dodge=False, ax=axes[1], width=0.65,
)
sns.stripplot(
    data=all_runs, x='architecture', y='test_f1_macro', order=order,
    color='#222222', size=5, jitter=0.08, ax=axes[1],
)
axes[1].set_title('Prueba · generalización final')
axes[1].set_xlabel('Arquitectura')
axes[1].set_ylabel('F1 macro')
axes[1].set_ylim(0, 1.03)
handles, labels = axes[1].get_legend_handles_labels()
axes[1].legend(handles[:2], labels[:2], title='Familia', loc='lower left', frameon=True)

fig.savefig(NB09_RESULTS / 'Figura_NB09_F1_multisemilla.png', dpi=300, bbox_inches='tight')
plt.show()

class_columns = [f'test_f1_{label}' for label in CLASSES]
class_means = all_runs.groupby('architecture')[class_columns].mean().reindex(order)
class_means.columns = CLASSES
class_means.to_csv(NB09_RESULTS / 'F1_test_por_clase_promedio.csv')

fig, ax = plt.subplots(figsize=(10.5, 5.5))
sns.heatmap(
    class_means, annot=True, fmt='.3f', cmap='YlGnBu',
    vmin=0, vmax=1, linewidths=0.5, linecolor='white', ax=ax,
)
ax.set_title('F1 promedio por clase en prueba · tres semillas')
ax.set_xlabel('Clase')
ax.set_ylabel('Arquitectura')
fig.savefig(NB09_RESULTS / 'Figura_NB09_F1_por_clase.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8.5, 5.8))
for _, row in aggregate.iterrows():
    ax.scatter(
        row['inference_ms_image_mean'], row['test_f1_macro_mean'],
        s=170, color=palette[row['family']], edgecolor='black', linewidth=0.7,
    )
    ax.annotate(
        row['architecture'],
        (row['inference_ms_image_mean'], row['test_f1_macro_mean']),
        xytext=(6, 5), textcoords='offset points', fontsize=9,
    )
ax.set_xlabel('Tiempo de inferencia promedio (ms/imagen)')
ax.set_ylabel('F1 macro promedio en prueba')
ax.set_title('Compromiso entre generalización y velocidad')
fig.savefig(NB09_RESULTS / 'Figura_NB09_F1_velocidad.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Selección y cierre


In [ ]:
winner = aggregate.iloc[0]
selection = {
    'selection_rule': 'Mayor F1 macro promedio de validación; prueba no utilizada para seleccionar.',
    'selected_model': winner['architecture'],
    'selected_val_f1_macro_mean': float(winner['val_f1_macro_mean']),
    'selected_test_f1_macro_mean': float(winner['test_f1_macro_mean']),
    'seeds': SEEDS,
    'comparison_scope': 'YOLO11 vs YOLO26; tamaños n, s y m; mismo split y protocolo.',
    'warning': 'La comparación corresponde a este dataset y no garantiza superioridad universal.',
}
(NB09_RESULTS / 'seleccion_modelo.json').write_text(
    json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8'
)

environment = {
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'python': sys.version,
    'platform': platform.platform(),
    'torch': torch.__version__,
    'ultralytics': ultralytics.__version__,
    'scikit_learn': sklearn.__version__,
    'gpu': GPU_NAME,
    'gpu_memory_gb': round(GPU_MEMORY_GB, 2),
    'models': [exp['weights'] for exp in EXPERIMENTS],
    'seeds': SEEDS,
    'imgsz': IMGSZ,
}
(NB09_RESULTS / 'entorno_ejecucion.json').write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8'
)

print('=' * 72)
print('NB09 FINALIZADO CORRECTAMENTE')
print('Modelo seleccionado por validación:', winner['architecture'])
print('Resultados:', NB09_RESULTS)
print('=' * 72)


## Resultado esperado

Al terminar debe aparecer **`NB09 FINALIZADO CORRECTAMENTE`**.  
Cada corrida terminada queda guardada en Drive. Si Colab se desconecta, vuelva a ejecutar el notebook desde el inicio: las corridas completas se detectan y no se repiten.
